## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [1]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# Skip periodic JSON/HTML diagnostics and per-frame logging for every run.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1"

# Apply the measured vLLM winner before any serving setup command runs.
PUBLIC25_VLLM_PROFILE_NAME = 'kv5-bf16-mtp3-c8-cg32'
PUBLIC25_VLLM_PROFILE_ENV = {
    "TAAF_VLLM_ENABLE_PREFIX_CACHING": "0",
    "TAAF_VLLM_KV_CACHE_DTYPE": "auto",
    "TAAF_VLLM_KV_CACHE_MEMORY_BYTES": "5368709120",
    "TAAF_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE": "32",
    "TAAF_VLLM_MAX_NUM_BATCHED_TOKENS": "8192",
    "TAAF_VLLM_MAX_NUM_SEQS": "8",
    "TAAF_VLLM_MTP_TOKENS": "3",
    "TAAF_VLLM_OMP_THREADS": "1"
}
for key, value in PUBLIC25_VLLM_PROFILE_ENV.items():
    os.environ[key] = value
print(
    f'PUBLIC25_VLLM_PROFILE name={PUBLIC25_VLLM_PROFILE_NAME} '
    f'env={json.dumps(PUBLIC25_VLLM_PROFILE_ENV, sort_keys=True)}',
    flush=True,
)
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

PUBLIC25_VLLM_PROFILE name=kv5-bf16-mtp3-c8-cg32 env={"TAAF_VLLM_ENABLE_PREFIX_CACHING": "0", "TAAF_VLLM_KV_CACHE_DTYPE": "auto", "TAAF_VLLM_KV_CACHE_MEMORY_BYTES": "5368709120", "TAAF_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE": "32", "TAAF_VLLM_MAX_NUM_BATCHED_TOKENS": "8192", "TAAF_VLLM_MAX_NUM_SEQS": "8", "TAAF_VLLM_MTP_TOKENS": "3", "TAAF_VLLM_OMP_THREADS": "1"}
taaf.kaggle: TRUE_SUBMISSION=False


## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [2]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

0

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [3]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1", "keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

taaf.kaggle: source bundle = /kaggle/input/datasets/keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1
taaf.kaggle: input paths = {"keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1": "/kaggle/input/datasets/keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1", "keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1": "/kaggle/input/datasets/keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1"}


## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands â€” installing wheels, fetching model weights, and so on.

In [4]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

taaf.kaggle: wrote /usr/local/lib/python3.12/dist-packages/taaf_kaggle_sources.pth (5 source roots)
taaf.kaggle: setup command: "$PYTHON" "$TAAF_KAGGLE_BUNDLE_DIR/serving_setup.py"
VLLM_SETUP_MODE fast TAAF_KAGGLE_FAST_START=1
PINNED_VLLM_RUNTIME_PATH /kaggle/input/datasets/keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1
PINNED_MODEL_PATH /kaggle/input/models/keithtyser/qwen3-8-flash-next-nvfp4/pytorch/radixark-modelopt-fp4/1
RUNTIME_LAYER_FAST_CHECK index=8 bytes=1374796713 manifest_sha256=f4f325a97c3e4739640b0e472a818bb299e7ce8e585bdbfe5586e9d3b5a2c475
RUNTIME_LAYER_FAST_CHECK index=17 bytes=4634843373 manifest_sha256=a5881cf32fc46f24318a6fc7015ccae863706075d4434d186b8b1b4e5bafb36e
RUNTIME_LAYER_FAST_CHECK index=18 bytes=1511319261 manifest_sha256=692ea0d28a87e9cea6db254c830f650f29804522045c9201f8ea63f30471b33b
RUNTIME_LAYER_FAST_CHECK index=21 bytes=61571140 manifest_sha256=651e9c6638c77d15bf671d9264badcba80ea551f84109de486427b87d2ddbf75
RUNTIME_LAYER_FAST_CHECK index=24 bytes=30

## 4A. P1 SPARSE — Prompt-truth subsystem (one subsystem on top of frozen B0)

Donor: `sonpham-org/arc-3` → `kaggle/experiments/sparse-deletion/build_bundles.py`
(blob `45f2ff9fca842a71d5214ca2efce5d92c6af9829`). Its control bundle is this notebook's
own `keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1`.

**P1 is frozen to exactly Arm B + Arm C:**
- Arm B: remove the false/hard prompt priors.
- Arm C: add the six possibility-phrased mechanics lines.

**ACTION7 is deliberately NOT changed in P1.** That is donor Arm H / a later P1b correctness arm.

The patch is installed before `benchmark_initial.pkl` is loaded, pins the exact B0 prompt and P1 prompt
by SHA-256, pins the runtime prompt budget to the B0 value (1024), verifies ACTION7 remains unexecutable
as in B0, and fails closed on drift. Model, serving, sampling, context, scheduler, tools, memory and run
geometry are not treatment variables.


In [5]:
# P1 SPARSE FINAL -- Prompt-truth subsystem: exactly donor Arm B + Arm C, prompt only.
# ACTION7 / serving / sampling / context / scheduler / memory / tools are intentionally untouched.
import difflib
import hashlib
import inspect
import json
import os
from pathlib import Path

import inference.agent.action_names as _duck_action_names
import inference.agent.prompts as _duck_prompts
import inference.agent.tool_agent as _duck_tool_agent

P1_NAME = "B0+P1_SPARSE"
P1_PATCH_VERSION = "p1-sparse-final-20260915"
P1_DONOR_BUILD_BUNDLES_BLOB_SHA = "45f2ff9fca842a71d5214ca2efce5d92c6af9829"
P1_EXPECTED_RUNTIME_TOOL_OUTPUT_TOKENS = 1024
P1_EXPECTED_CONTROL_PROMPT_SHA256 = "d091a36db1ccf79e5caa35b2cd31949370ef95810d00d713cc98eb27a3cd60a6"
P1_EXPECTED_CONTROL_PROMPT_CHARS = 12681
P1_EXPECTED_PROMPT_SHA256 = "620c352791957f63c31338a71bc28855119f9b557cbc1f88a23aa0182cbbfe6e"
P1_EXPECTED_PROMPT_CHARS = 13032
P1_ALLOW_BASE_DRIFT = False

HUD_PARAGRAPHS = (
    "- In many games, a long horizontal or vertical line near an edge is a timer or remaining-steps bar. "
    "It often shrinks or changes each step. If you identify such a bar, do not get distracted by it or treat it "
    "as core gameplay state unless there is concrete evidence that it interacts with the puzzle mechanics.\n"
    "A common failure mode is to mistake a segmented edge bar for clickable puzzle pieces. If a repeated strip of "
    "small blocks sits flush against the top, bottom, left, or right border and actions only change that strip "
    "while the interior board stays the same, classify it as HUD/timer state, not as an object to click through "
    "segment by segment. DON'T DO THIS!\n"
)

# Exact effective Arm-B edits to the assembled B0 prompt.
P1_DELETIONS = (
    ("You are a coding agent solving a grid-based puzzle game.",
     "You are a coding agent solving a grid-based game."),
    ("- You are solving a multi-level grid puzzle game. \n",
     "- You are solving a multi-level grid game. \n"),
    ("boards are presented as 64 x 64 color grids",
     "boards are presented as color grids"),
    ("- Some games are logic or layout puzzles with no explicit player avatar",
     "- Some games are logic or layout games with no explicit player avatar"),
    (HUD_PARAGRAPHS, ""),
)

# Exact runtime text produced by donor Arm C.
P1_MECHANICS_LINES = (
    "- The visible board may be a window onto a larger world, and that window can move. If the whole scene "
    "shifts at once, the frame moved, not the contents; the same row/col may not mean the same place as it did "
    "last turn.\n"
    "- What you control may change. An action may hand control to a different object instead of acting on the "
    "board, so do not assume there is exactly one controllable thing for the whole game.\n"
    "- State that decides the outcome may not be drawn on the board: what is being carried, what was collected, "
    "or what happened on an earlier attempt can change what the same action does now.\n"
    "- Order can matter as much as position. A set of things in the right places may still be wrong if they are "
    "read in a sequence.\n"
    "- Something that looks solved can come un-solved by a later action, and parts of the board may act on their "
    "own between your actions, with or against you.\n"
    "- Control may be indirect: you may move something that drags what you care about, or set values that are "
    "read together as a code rather than acting one at a time.\n"
)

P1_LEGEND_LINE_PREFIX = "- Color legend: "
P1_MECHANICS_MARKER = "- The visible board may be a window onto a larger world"
P1_DELETED_PROBES = ("DON'T DO THIS", "remaining-steps bar", "64 x 64", "puzzle")
P1_FORBIDDEN_OTHER_ARM_MARKERS = (
    "Exploration is for building one hypothesis",   # donor F
    "On the first turn only,",                      # donor E
    "measuring instruments, not the board",         # donor G
    "valid, executable game action",                # donor H prompt line
    "Prediction verification discipline",           # user's earlier VISTA addendum
)

def _p1_sha(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def _p1_transform(text: str) -> str:
    out = text
    for old, new in P1_DELETIONS:
        n = out.count(old)
        if n != 1:
            raise RuntimeError(f"P1 edit target occurs {n} times (expected 1): {old[:80]!r}")
        out = out.replace(old, new, 1)
    if out.count(P1_LEGEND_LINE_PREFIX) != 1:
        raise RuntimeError("P1 could not locate a unique Color legend line.")
    start = out.index(P1_LEGEND_LINE_PREFIX)
    end = out.index("\n", start) + 1
    return out[:end] + P1_MECHANICS_LINES + out[end:]

def _p1_semantic_checks(prompt: str, *, label: str) -> None:
    bad = [probe for probe in P1_DELETED_PROBES if probe in prompt]
    if bad:
        raise RuntimeError(f"P1 {label}: deleted probes remain: {bad}")
    foreign = [m for m in P1_FORBIDDEN_OTHER_ARM_MARKERS if m in prompt]
    if foreign:
        raise RuntimeError(f"P1 {label}: foreign-arm markers present: {foreign}")
    if prompt.count(P1_MECHANICS_MARKER) != 1:
        raise RuntimeError(f"P1 {label}: mechanics marker count != 1.")

def _module_under_bundle(module) -> bool:
    p = Path(module.__file__).resolve()
    try:
        p.relative_to(Path(BUNDLE_DIR).resolve())
        return True
    except ValueError:
        return False

# Ensure the imported Duck code is the attached B0 bundle, not another installed copy.
for _m in (_duck_tool_agent, _duck_prompts, _duck_action_names):
    if not _module_under_bundle(_m):
        raise RuntimeError(f"P1 imported {_m.__name__} from outside B0 bundle: {_m.__file__}")

# Freeze the actual B0 runtime prompt budget before gameplay.
_runtime_tool_output_tokens = int(
    getattr(_duck_tool_agent, "_LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS",
            os.environ.get("LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS", "0"))
)
if _runtime_tool_output_tokens != P1_EXPECTED_RUNTIME_TOOL_OUTPUT_TOKENS:
    raise RuntimeError(
        "P1 runtime drift: LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS="
        f"{_runtime_tool_output_tokens}, expected {P1_EXPECTED_RUNTIME_TOOL_OUTPUT_TOKENS}."
    )

# ACTION7 is a separate donor arm. P1 must preserve the B0 defect exactly.
_action7_before = _duck_action_names.to_engine_action("ACTION7")
if _action7_before is not None:
    raise RuntimeError(
        f"P1 base action-map drift: ACTION7 resolves to {_action7_before!r}; expected None in frozen B0."
    )

_installed = getattr(_duck_tool_agent, "_P1_SPARSE_INSTALLED", None)
_builder = (
    _duck_tool_agent._P1_SPARSE_ORIGINAL_BUILD_SYSTEM_PROMPT
    if _installed else _duck_tool_agent._build_system_prompt
)
_builder_source = inspect.getsource(_builder)
for _frag in (
    'prompt = "You are a coding agent solving a grid-based puzzle game."',
    "prompt += GAME_OVERVIEW_ADDENDUM",
    "prompt += STRUCTURED_RUNTIME_STATE_ADDENDUM",
    "prompt += VISUAL_GAME_ADDENDUM",
    "prompt += PYTHON_ADDENDUM",
    "prompt += COMPACT_TOOL_SESSION_ADDENDUM.format",
):
    if _frag not in _builder_source:
        raise RuntimeError(f"P1 unexpected Duck prompt builder; missing fragment: {_frag!r}")

if _installed:
    # Idempotent only for this frozen treatment; changing it requires a fresh kernel.
    if _installed.get("version") != P1_PATCH_VERSION:
        raise RuntimeError(f"P1 already installed with another version: {_installed!r}")
    _live = _duck_tool_agent._build_system_prompt(
        tool_output_tokens=P1_EXPECTED_RUNTIME_TOOL_OUTPUT_TOKENS
    )
    if len(_live) != P1_EXPECTED_PROMPT_CHARS or _p1_sha(_live) != P1_EXPECTED_PROMPT_SHA256:
        raise RuntimeError("P1 re-run check failed: installed prompt drifted.")
    _p1_semantic_checks(_live, label="re-run")
    print("P1_SPARSE already installed; frozen treatment verified.", flush=True)
else:
    _original = _duck_tool_agent._build_system_prompt
    _base = _original(tool_output_tokens=P1_EXPECTED_RUNTIME_TOOL_OUTPUT_TOKENS)
    _base_sha = _p1_sha(_base)
    _base_is_b0 = (
        len(_base) == P1_EXPECTED_CONTROL_PROMPT_CHARS
        and _base_sha == P1_EXPECTED_CONTROL_PROMPT_SHA256
    )
    if not _base_is_b0 and not P1_ALLOW_BASE_DRIFT:
        raise RuntimeError(
            f"P1 base prompt drift: chars={len(_base)} sha={_base_sha}; "
            f"expected {P1_EXPECTED_CONTROL_PROMPT_CHARS}/{P1_EXPECTED_CONTROL_PROMPT_SHA256}."
        )
    for _probe in P1_DELETED_PROBES:
        if _probe not in _base:
            raise RuntimeError(f"P1 base prompt lacks expected donor deletion probe {_probe!r}.")
    if P1_MECHANICS_MARKER in _base:
        raise RuntimeError("P1 base already contains Arm-C mechanics block.")
    for _marker in P1_FORBIDDEN_OTHER_ARM_MARKERS:
        if _marker in _base:
            raise RuntimeError(f"P1 base carries foreign-arm marker {_marker!r}.")

    _patched = _p1_transform(_base)
    _p1_semantic_checks(_patched, label="patched")
    _patched_sha = _p1_sha(_patched)
    if len(_patched) != P1_EXPECTED_PROMPT_CHARS or _patched_sha != P1_EXPECTED_PROMPT_SHA256:
        raise RuntimeError(
            f"P1 prompt mismatch: chars={len(_patched)} sha={_patched_sha}; "
            f"expected {P1_EXPECTED_PROMPT_CHARS}/{P1_EXPECTED_PROMPT_SHA256}."
        )

    def _p1_build_system_prompt(*, tool_output_tokens: int, _orig=_original) -> str:
        return _p1_transform(_orig(tool_output_tokens=tool_output_tokens))

    _duck_tool_agent._P1_SPARSE_ORIGINAL_BUILD_SYSTEM_PROMPT = _original
    _duck_tool_agent._build_system_prompt = _p1_build_system_prompt
    _duck_tool_agent._P1_SPARSE_INSTALLED = {
        "version": P1_PATCH_VERSION,
        "prompt_sha256": _patched_sha,
        "prompt_chars": len(_patched),
    }

    _live = _duck_tool_agent._build_system_prompt(
        tool_output_tokens=P1_EXPECTED_RUNTIME_TOOL_OUTPUT_TOKENS
    )
    if _live != _patched:
        raise RuntimeError("P1 live wrapper differs from verified patched prompt.")

    def _file_record(module):
        p = Path(module.__file__).resolve()
        return {
            "path": str(p),
            "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            "bytes": p.stat().st_size,
        }

    patch_record = {
        "experiment": P1_NAME,
        "patch_version": P1_PATCH_VERSION,
        "treatment": "donor Arm B + Arm C only",
        "donor_build_bundles_blob_sha": P1_DONOR_BUILD_BUNDLES_BLOB_SHA,
        "action7_change_included": False,
        "base_is_frozen_b0": _base_is_b0,
        "runtime_tool_output_tokens": _runtime_tool_output_tokens,
        "control_prompt": {"chars": len(_base), "sha256": _base_sha},
        "p1_prompt": {"chars": len(_patched), "sha256": _patched_sha},
        "chars_delta": len(_patched) - len(_base),
        "probes_present_in_base": {p: p in _base for p in P1_DELETED_PROBES},
        "probes_absent_after_patch": {p: p not in _patched for p in P1_DELETED_PROBES},
        "mechanics_marker_count": _patched.count(P1_MECHANICS_MARKER),
        "foreign_arm_markers_absent": {m: m not in _patched for m in P1_FORBIDDEN_OTHER_ARM_MARKERS},
        "action7_to_engine_before": _action7_before,
        "action7_to_engine_after": _duck_action_names.to_engine_action("ACTION7"),
        "sources": {
            "tool_agent": _file_record(_duck_tool_agent),
            "prompts": _file_record(_duck_prompts),
            "action_names": _file_record(_duck_action_names),
        },
    }
    if patch_record["action7_to_engine_after"] is not None:
        raise RuntimeError("P1 accidentally changed ACTION7; treatment is contaminated.")

    (WORKING_DIR / "sparse_p1_patch.json").write_text(
        json.dumps(patch_record, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    (WORKING_DIR / "sparse_p1_system_prompt_base.txt").write_text(_base, encoding="utf-8")
    (WORKING_DIR / "sparse_p1_system_prompt_patched.txt").write_text(_patched, encoding="utf-8")
    (WORKING_DIR / "sparse_p1_prompt_diff.txt").write_text(
        "".join(difflib.unified_diff(
            _base.splitlines(True), _patched.splitlines(True),
            fromfile="B0_system_prompt", tofile="P1_SPARSE_system_prompt", n=1
        )),
        encoding="utf-8",
    )

    manifest = f"""# P1 SPARSE FINAL

- Base notebook SHA-256: 21c99f92bb6ecdbb97fadc66c15b10ea24a13bcb1958b06774ba522e6f1a9f6e
- Donor build_bundles.py blob: {P1_DONOR_BUILD_BUNDLES_BLOB_SHA}
- Treatment: Arm B deletions + Arm C mechanics block.
- ACTION7: unchanged, expected `to_engine_action("ACTION7") is None`.
- Control prompt: {len(_base)} chars / {_base_sha}
- P1 prompt: {len(_patched)} chars / {_patched_sha}
- Runtime tool-output tokens: {_runtime_tool_output_tokens}

No model, vLLM, context, sampling, scheduler, tool, memory, action-cap, game-set or timing treatment is added by P1.
"""
    (WORKING_DIR / "sparse_p1_MANIFEST.md").write_text(manifest, encoding="utf-8")

    print(f"ARM_PROVENANCE arm={P1_NAME}", flush=True)
    print(f"ARM_PROVENANCE base_prompt_chars={len(_base)} sha256={_base_sha} frozen_b0={_base_is_b0}")
    print(f"ARM_PROVENANCE system_prompt_chars={len(_patched)} sha256={_patched_sha}")
    print("ARM_PROVENANCE action7_change_included=False action7_to_engine=None")
    print(f"P1_SPARSE installed version={P1_PATCH_VERSION}", flush=True)


ARM_PROVENANCE arm=B0+P1_SPARSE
ARM_PROVENANCE base_prompt_chars=12681 sha256=d091a36db1ccf79e5caa35b2cd31949370ef95810d00d713cc98eb27a3cd60a6 frozen_b0=True
ARM_PROVENANCE system_prompt_chars=13032 sha256=620c352791957f63c31338a71bc28855119f9b557cbc1f88a23aa0182cbbfe6e
ARM_PROVENANCE action7_change_included=False action7_to_engine=None
P1_SPARSE installed version=p1-sparse-final-20260915


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [6]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 6. Customization hook

Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts â€” the safe place
for one-off experiments once the deployed bundle has loaded.

In [7]:
# Exact public-25 and competition settings.
bm.solver.max_runtime_s_per_game = 7920.0
bm.solver.analyzer_timeout = 900.0
bm.solver.concurrency = 28
bm.solver.max_actions_per_game = None
bm.solver.save_request_logs = False
if float(getattr(target, 'max_runtime_s', 0.0) or 0.0) != 32400.0:
    raise RuntimeError(
        f'Expected the 32400-second notebook budget, got {target.max_runtime_s!r}.'
    )
print(
    f'PUBLIC25_SETTINGS budget_s={bm.solver.max_runtime_s_per_game} '
    f'concurrency={bm.solver.concurrency} analyzer_timeout={bm.solver.analyzer_timeout} '
    f'action_cap={bm.solver.max_actions_per_game} request_logs={bm.solver.save_request_logs}',
    flush=True,
)


PUBLIC25_SETTINGS budget_s=7920.0 concurrency=28 analyzer_timeout=900.0 action_cap=None request_logs=False


## 6A. P1 fail-closed runtime preflight

Instrumentation only; it does not change gameplay. It proves the live run still has the frozen B0
serving/sampling/solver contract and the exact P1 prompt before any game starts. Any drift raises here,
rather than invalidating a two-hour run afterwards.


In [8]:
# P1 fail-closed runtime preflight. NO gameplay/model request is made here.
P1_EVIDENCE_SCHEMA_VERSION = "p1-evidence-v2-20260915"
import hashlib
import json
from urllib.request import urlopen

import inference.agent.action_names as _p1_an
import inference.agent.tool_agent as _p1_ta

_P1_EXPECTED_PROFILE = {
    "TAAF_VLLM_ENABLE_PREFIX_CACHING": "0",
    "TAAF_VLLM_KV_CACHE_DTYPE": "auto",
    "TAAF_VLLM_KV_CACHE_MEMORY_BYTES": "5368709120",
    "TAAF_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE": "32",
    "TAAF_VLLM_MAX_NUM_BATCHED_TOKENS": "8192",
    "TAAF_VLLM_MAX_NUM_SEQS": "8",
    "TAAF_VLLM_MTP_TOKENS": "3",
    "TAAF_VLLM_OMP_THREADS": "1",
}
_live_prompt = _p1_ta._build_system_prompt(tool_output_tokens=P1_EXPECTED_RUNTIME_TOOL_OUTPUT_TOKENS)
_live_sha = hashlib.sha256(_live_prompt.encode("utf-8")).hexdigest()

try:
    with urlopen("http://127.0.0.1:1234/v1/models", timeout=10) as _resp:
        _models_payload = json.loads(_resp.read().decode("utf-8", "replace"))
except Exception as _exc:
    raise RuntimeError(f"P1 preflight could not query vLLM /v1/models: {_exc!r}") from _exc

_served_model_ids = [
    str(item.get("id")) for item in (_models_payload.get("data") or [])
    if isinstance(item, dict) and item.get("id")
]
_expected_model = "Qwen/Qwen3.8-Flash-Next-NVFP4"

_checks = {
    "profile_name": PUBLIC25_VLLM_PROFILE_NAME == "kv5-bf16-mtp3-c8-cg32",
    "profile_dict": PUBLIC25_VLLM_PROFILE_ENV == _P1_EXPECTED_PROFILE,
    "profile_env": all(os.environ.get(k) == v for k, v in _P1_EXPECTED_PROFILE.items()),
    "context_window": int(getattr(_p1_ta, "_LOCAL_ANALYZER_CONTEXT_WINDOW", -1)) == 32768,
    "tool_output_tokens": int(getattr(_p1_ta, "_LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS", -1)) == 1024,
    "temperature": float(getattr(_p1_ta, "_LOCAL_ANALYZER_TEMPERATURE", -1)) == 0.6,
    "top_p": float(getattr(_p1_ta, "_LOCAL_ANALYZER_TOP_P", -1)) == 0.95,
    "top_k": int(getattr(_p1_ta, "_LOCAL_ANALYZER_TOP_K", -1)) == 20,
    "thinking": bool(getattr(_p1_ta, "_LOCAL_ANALYZER_ENABLE_THINKING", False)) is True,
    "served_model": _expected_model in _served_model_ids,
    "solver_runtime": float(bm.solver.max_runtime_s_per_game) == 7920.0,
    "solver_timeout": float(bm.solver.analyzer_timeout) == 900.0,
    "solver_concurrency": int(bm.solver.concurrency) == 28,
    "solver_action_cap_none": bm.solver.max_actions_per_game is None,
    "request_logs_off": bool(bm.solver.save_request_logs) is False,
    "notebook_budget": float(getattr(target, "max_runtime_s", 0.0) or 0.0) == 32400.0,
    "p1_installed": getattr(_p1_ta, "_P1_SPARSE_INSTALLED", {}).get("version") == P1_PATCH_VERSION,
    "p1_prompt_chars": len(_live_prompt) == P1_EXPECTED_PROMPT_CHARS,
    "p1_prompt_sha256": _live_sha == P1_EXPECTED_PROMPT_SHA256,
    "action7_unchanged": _p1_an.to_engine_action("ACTION7") is None,
}
_failed = [k for k, v in _checks.items() if v is not True]
_preflight = {
    "experiment": P1_NAME,
    "patch_version": P1_PATCH_VERSION,
    "checks": _checks,
    "failed": _failed,
    "served_model_ids": _served_model_ids,
    "expected_model": _expected_model,
    "system_prompt_chars": len(_live_prompt),
    "system_prompt_sha256": _live_sha,
    "solver": {
        "max_runtime_s_per_game": bm.solver.max_runtime_s_per_game,
        "analyzer_timeout": bm.solver.analyzer_timeout,
        "concurrency": bm.solver.concurrency,
        "max_actions_per_game": bm.solver.max_actions_per_game,
        "save_request_logs": bm.solver.save_request_logs,
    },
}
_preflight["evidence_schema_version"] = P1_EVIDENCE_SCHEMA_VERSION
_preflight["notebook_start_epoch"] = float(NOTEBOOK_START_EPOCH)
_preflight["preflight_completed_epoch"] = float(time.time())
_preflight["startup_elapsed_s_at_preflight"] = round(
    _preflight["preflight_completed_epoch"] - _preflight["notebook_start_epoch"], 6
)
(WORKING_DIR / "sparse_p1_preflight.json").write_text(
    json.dumps(_preflight, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
if _failed:
    raise RuntimeError(f"P1 runtime preflight failed: {_failed}")
print(
    "P1_PREFLIGHT_PASS "
    f"prompt={len(_live_prompt)}/{_live_sha[:16]} model={_expected_model} "
    "ctx=32768 mtp=3 max_seqs=8 concurrency=28 action_cap=None",
    flush=True,
)


P1_PREFLIGHT_PASS prompt=13032/620c352791957f63 model=Qwen/Qwen3.8-Flash-Next-NVFP4 ctx=32768 mtp=3 max_seqs=8 concurrency=28 action_cap=None


## 7. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and
play the **live competition Arcade**. Otherwise â€” an interactive "Save & Run" â€” play the
competition's **bundled environment files offline**, with no gateway required, so the notebook
runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [9]:
# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Skip pre-run display and git-status copies; the staged identity pins the run.

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

PUBLIC_GAME_IDS = tuple([
    "tn36-ef4dde99",
    "lf52-271a04aa",
    "cn04-2fe56bfb",
    "bp35-0a0ad940",
    "wa30-ee6fef47",
    "lp85-305b61c3",
    "r11l-495a7899",
    "tu93-0768757b",
    "sp80-589a99af",
    "m0r0-492f87ba",
    "vc33-5430563c",
    "ar25-0c556536",
    "ka59-38d34dbb",
    "sc25-635fd71a",
    "sk48-d8078629",
    "dc22-fdcac232",
    "cd82-fb555c5d",
    "ft09-0d8bbf25",
    "g50t-5849a774",
    "ls20-9607627b",
    "re86-8af5384d",
    "s5i5-18d95033",
    "sb26-7fbdac44",
    "su15-1944f8ab",
    "tr87-cd924810"
])

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
    offline_games = _offline_games(competition_env_files)
    offline_by_id = {game.env_name: game for game in offline_games}
    if len(offline_by_id) != len(offline_games):
        raise RuntimeError('The offline public game list contains duplicate IDs.')
    missing = sorted(set(PUBLIC_GAME_IDS) - set(offline_by_id))
    extra = sorted(set(offline_by_id) - set(PUBLIC_GAME_IDS))
    if missing or extra:
        raise RuntimeError(
            f'Offline public game set changed; missing={missing}, extra={extra}.'
        )
    bm.games = [offline_by_id[game_id] for game_id in PUBLIC_GAME_IDS]
    if len(bm.games) != 25:
        raise RuntimeError(f'Expected 25 public games, got {len(bm.games)}.')
    print(f'PUBLIC25_SELECTION games={len(bm.games)} passes=1', flush=True)

bm.n_passes = 1
bm.game_weights = None

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
if budget <= 600.0:
    raise RuntimeError(f'Notebook budget is too small for the teardown reserve: {budget}.')
soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(
    seconds=budget - 600.0
)

# Start recovery only after setup readiness and all run gates pass.
if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))
import vllm_server_watchdog as vllm_watchdog

vllm_watchdog_setup = vllm_watchdog.load_setup(BUNDLE_DIR / 'serving_setup.py')
vllm_watchdog.start_background(
    vllm_watchdog_setup,
    vllm_watchdog.WatchdogConfig(
        interval_seconds=15.0,
        request_timeout_seconds=5,
        failure_threshold=4,
        max_restart_attempts=2,
    ),
)

# Play the benchmark; watchdog stop and teardown run even if it raises.
try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=True)
    bm._save_json()
    if not TRUE_SUBMISSION:
        # Kaggle Save & Run expects this valid placeholder after an offline run.
        # A real competition rerun uses the live gateway and never enters this branch.
        import pandas as pd

        pd.DataFrame(
            [["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        ).to_parquet(WORKING_DIR / "submission.parquet", index=False)

        # Check terminal coverage, then call the frozen scorer once.
        # This path does not render HTML.
        public_runs = list(bm.game_runs)
        public_run_ids = [run.game_id for run in public_runs]
        if len(public_runs) != 25 or public_run_ids != list(PUBLIC_GAME_IDS):
            raise RuntimeError(
                f'Public run coverage changed: count={len(public_runs)} ids={public_run_ids}.'
            )
        unfinished = [
            (run.game_id, run.state, run.final_score)
            for run in public_runs
            if run.state not in {'won', 'gave_up', 'cancelled'}
            or run.final_score is None
        ]
        if unfinished:
            raise RuntimeError(f'Public runs did not finalize cleanly: {unfinished}.')
        crashed = [run.game_id for run in public_runs if run.state == 'crashed']
        if crashed:
            raise RuntimeError(f'Public runs crashed: {crashed}.')
        total_actions = sum(len(run.history) for run in public_runs)
        if total_actions <= 0:
            raise RuntimeError('Public runs produced no actions.')

        from inference.tools.eval import evaluate_runs, save_score_file

        score_summary = evaluate_runs([WORKING_DIR])
        score_path = save_score_file(
            score_summary,
            run_dirs=[WORKING_DIR],
            output_path=WORKING_DIR / "score.json",
        )
        if Path(score_path) != WORKING_DIR / 'score.json' or not Path(score_path).is_file():
            raise RuntimeError(f'Frozen scorer did not write score.json: {score_path}.')
        print(
            f'PUBLIC25_AUDIT runs=25 actions={total_actions} score_path={score_path}',
            flush=True,
        )
finally:
    try:
        vllm_watchdog.stop_background(timeout_seconds=15.0)
    finally:
        for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
            print(f"taaf.kaggle: teardown command: {command}", flush=True)
            subprocess.run(
                command,
                shell=True,
                check=False,
                cwd=WORKING_DIR,
                env=_command_env(),
                timeout=30.0,
            )


PUBLIC25_SELECTION games=25 passes=1
benchmark: duck-harness-kaggle
solver:    duck-harness
games:     25
passes:    1
runs:      25 (won: 0)
started:   2026-09-15 11:03:53
ended:     in progress
mean score:    0.33
median score:  0.00
total actions: 263
total tokens:  148477
generated tokens/sec: 246.98 (job wallclock)
total wallclock: 11541.2s

per-game (mean across passes):
  ar25-0c556536: score=2.78, levels=1.0/8, actions=19, tokens=6831
  bp35-0a0ad940: score=0.00, levels=0.0/9, actions=4, tokens=5140
  cd82-fb555c5d: score=0.00, levels=0.0/6, actions=6, tokens=6143
  cn04-2fe56bfb: score=0.00, levels=0.0/6, actions=4, tokens=5132
  dc22-fdcac232: score=0.00, levels=0.0/6, actions=14, tokens=4610
  ft09-0d8bbf25: score=0.00, levels=0.0/6, actions=4, tokens=7204
  g50t-5849a774: score=0.00, levels=0.0/7, actions=2, tokens=9402
  ka59-38d34dbb: score=0.00, levels=0.0/7, actions=3, tokens=5588
  lf52-271a04aa: score=0.00, levels=0.0/10, actions=8, tokens=5725
  lp85-305b61c3: score=

<python_tool>:7: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?


benchmark: duck-harness-kaggle
solver:    duck-harness
games:     25
passes:    1
runs:      25 (won: 0)
started:   2026-09-15 11:03:53
ended:     in progress
mean score:    5.56
median score:  1.82
total actions: 2274
total tokens:  1228022
generated tokens/sec: 227.35 (job wallclock)
total wallclock: 120010.0s

per-game (mean across passes):
  ar25-0c556536: score=8.33, levels=2.0/8, actions=141, tokens=43311
  bp35-0a0ad940: score=0.00, levels=0.0/9, actions=36, tokens=33134
  cd82-fb555c5d: score=0.00, levels=0.0/6, actions=45, tokens=52937
  cn04-2fe56bfb: score=4.76, levels=1.0/6, actions=24, tokens=23766
  dc22-fdcac232: score=1.66, levels=1.0/6, actions=102, tokens=53565
  ft09-0d8bbf25: score=47.62, levels=4.0/6, actions=59, tokens=39266
  g50t-5849a774: score=0.00, levels=0.0/7, actions=25, tokens=65664
  ka59-38d34dbb: score=3.57, levels=1.0/7, actions=36, tokens=47873
  lf52-271a04aa: score=1.82, levels=1.0/10, actions=111, tokens=41060
  lp85-305b61c3: score=16.67, levels=

analyzer request failed at action 66: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=119.10955909800032)
analyzer request failed at action 45: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=83.6043516840009)
analyzer request failed at action 61: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=23.70357047499965)
analyzer request failed at action 143: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=98.88888757700079)
analyzer request failed at action 184: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=66.34840113599967)
analyzer request failed at action 159: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=88.10458519300028)
analyzer request failed at action 79: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=18.368281082000067)
analyzer request failed at action 264: HTTPC

[finished] bp35-0a0ad940 state=gave_up level=1/9 score=0.44 actions=65 tokens=67616 per-level=47/21,18/48,0/44,0/38,0/33,0/87,0/86,0/131,0/163 note="tokens=68119"
[finished] m0r0-492f87ba state=gave_up level=0/6 score=0.00 actions=44 tokens=64806 per-level=44/30,0/111,0/203,0/26,0/500,0/237 note="tokens=76481"
[finished] cn04-2fe56bfb state=gave_up level=1/6 score=4.76 actions=60 tokens=75981 per-level=14/29,46/54,0/85,0/300,0/208,0/113 note="tokens=75981"
[finished] lf52-271a04aa state=gave_up level=1/10 score=1.82 actions=142 tokens=53177 per-level=14/32,128/81,0/60,0/71,0/205,0/148,0/244,0/109,0/164,0/225 note="tokens=63402"
[finished] wa30-ee6fef47 state=gave_up level=1/9 score=2.22 actions=183 tokens=63594 per-level=57/71,126/119,0/183,0/98,0/368,0/68,0/79,0/442,0/415 note="tokens=63594"
[finished] tu93-0768757b state=gave_up level=2/9 score=3.18 actions=158 tokens=114391 per-level=49/19,20/16,89/34,0/42,0/123,0/80,0/14,0/23,0/111 note="tokens=115355"
[finished] g50t-5849a774 stat

analyzer request failed at action 190: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=99.08249125700058)


[finished] r11l-495a7899 state=gave_up level=2/6 score=11.96 actions=44 tokens=90090 per-level=4/22,40/33,0/51,0/26,0/52,0/49 note="tokens=91116"
[finished] ar25-0c556536 state=gave_up level=3/8 score=12.36 actions=189 tokens=74205 per-level=19/32,23/50,130/75,17/37,0/89,0/159,0/233,0/73 note="tokens=74205"


analyzer request failed at action 150: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=127.42957076099992)
analyzer request failed at action 263: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=63.63970525399964)


[finished] sk48-d8078629 state=gave_up level=0/8 score=0.00 actions=149 tokens=107413 per-level=149/61,0/177,0/101,0/103,0/230,0/181,0/125,0/92 note="tokens=109652"
[finished] sb26-7fbdac44 state=gave_up level=1/8 score=2.78 actions=262 tokens=39759 per-level=9/18,253/28,0/18,0/19,0/31,0/23,0/58,0/18 note="tokens=39759"
benchmark: duck-harness-kaggle
solver:    duck-harness
games:     25
passes:    1
runs:      25 (won: 0)
started:   2026-09-15 11:03:53
ended:     2026-09-15 13:15:54
duration:  2h 12m 1s
mean score:    6.91
median score:  2.78
total actions: 3882
total tokens:  1912454
generated tokens/sec: 241.42 (job wallclock)
total wallclock: 198011.5s

per-game (mean across passes):
  ar25-0c556536: score=12.36, levels=3.0/8, actions=189, tokens=74205
  bp35-0a0ad940: score=0.44, levels=1.0/9, actions=65, tokens=67616
  cd82-fb555c5d: score=0.00, levels=0.0/6, actions=77, tokens=91977
  cn04-2fe56bfb: score=4.76, levels=1.0/6, actions=60, tokens=75981
  dc22-fdcac232: score=1.66, 

Traceback (most recent call last):
  File "/kaggle/input/datasets/keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1/serving_teardown.py", line 1503, in <module>
    main()
  File "/kaggle/input/datasets/keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1/serving_teardown.py", line 1494, in main
    raise RuntimeError("vLLM teardown did not reach the bounded terminal gate")
RuntimeError: vLLM teardown did not reach the bounded terminal gate


## 8. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [10]:
# Minimal diagnostics are enabled; skip post-run HTML rendering.


## 9. P1 SPARSE forensic post-run audit

Read-only and **after gameplay/teardown**, so it cannot reduce the solver's time budget. Besides score and
prompt attribution, it writes runtime/environment/source fingerprints, runtime-health counters, efficiency
telemetry, and timing. Raw notebook cell stdout is not tee'd because doing so would alter the execution path;
the important configuration is mirrored structurally in JSON, and the executed Kaggle notebook remains a
supplement if a cell itself errors.


In [11]:
# P1 SPARSE EVIDENCE V2 forensic post-run audit.
# Runs only after benchmark + watchdog stop + teardown. Read-only; failures are non-fatal.
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import re
import statistics
import subprocess
import sys
import sysconfig
import time
from collections import Counter, defaultdict
from pathlib import Path

P1_EVIDENCE_SCHEMA_VERSION = "p1-evidence-v2-20260915"
B0_REF = {
    "ft09": (45.14, 4), "re86": (27.78, 4), "vc33": (19.84, 3), "lp85": (16.67, 3), "cd82": (14.29, 2),
    "ar25": (7.96, 2), "m0r0": (4.76, 1), "r11l": (4.76, 1), "ka59": (3.57, 1), "ls20": (3.57, 1),
    "sb26": (2.78, 1), "tr87": (2.47, 1), "s5i5": (2.30, 1), "su15": (2.22, 1), "lf52": (1.82, 1),
    "tu93": (1.00, 2), "dc22": (0.66, 1), "cn04": (0.42, 1), "bp35": (0.34, 1), "g50t": (0.0, 0),
    "sc25": (0.0, 0), "sk48": (0.0, 0), "sp80": (0.0, 0), "tn36": (0.0, 0), "wa30": (0.0, 0),
}
SPARSE_TARGET_GAMES = ("g50t", "lf52", "ls20", "bp35", "wa30", "tn36", "sk48")
summary = {
    "experiment": P1_NAME,
    "patch_version": P1_PATCH_VERSION,
    "evidence_schema_version": P1_EVIDENCE_SCHEMA_VERSION,
    "sections": {},
    "warnings": [],
}

def _section(name, fn):
    try:
        summary["sections"][name] = fn()
    except Exception as exc:
        summary["warnings"].append(f"{name}: {type(exc).__name__}: {exc}")

def _write_json(name, obj):
    (WORKING_DIR / name).write_text(
        json.dumps(obj, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8"
    )

def _sha_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def _sha_text(text: str) -> str:
    return _sha_bytes(text.encode("utf-8"))

def _sha_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _safe_float(value):
    try:
        return round(float(value), 6)
    except Exception:
        return None

def _prompt_core(text: str) -> str:
    return text.rstrip("\n") + "\n"

def _prompt_log_render(text: str) -> str:
    # Duck's prompt log rendering removes blank lines from normalized message content.
    return "\n".join(line for line in text.splitlines() if line.strip()).strip()

def _system_blocks_from_transcript(text: str):
    blocks = []
    marker = "[SYSTEM PROMPT]\n"
    for part in text.split(marker)[1:]:
        block = part.split("\n\n[USER PROMPT]\n", 1)[0]
        if block:
            blocks.append(_prompt_core(block))
    return blocks

def _solver_note_tokens(note):
    m = re.search(r"(?:^|\b)tokens=(\d+)", str(note or ""))
    return int(m.group(1)) if m else None

def _action_id(entry):
    if not isinstance(entry, dict):
        return None
    a = entry.get("action")
    if isinstance(a, dict):
        return str(a.get("name") or a.get("action") or a.get("id") or "")
    if isinstance(a, str):
        return a
    return None

def _read_json(path: Path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def _games_section():
    bench = json.loads((WORKING_DIR / "benchmark.json").read_text(encoding="utf-8"))
    rows = []
    for run in bench.get("game_runs", []):
        gid = str(run.get("game_id", ""))
        prefix = gid[:4]
        hist = run.get("history") or []
        ids = Counter(_action_id(h) for h in hist)
        score = float(run.get("final_score") or 0.0)
        levels = int(run.get("levels_completed") or 0)
        solver_note = str(run.get("solver_note") or "")
        reported_tokens = _solver_note_tokens(solver_note)
        history_generated_tokens = sum(
            int(h.get("generated_tokens") or 0) for h in hist if isinstance(h, dict)
        )
        ref = B0_REF.get(prefix)
        rows.append({
            "game": gid, "prefix": prefix, "score": round(score, 6), "levels": levels,
            "of": int(run.get("number_of_levels") or 0), "actions": len(hist),
            "reported_generated_tokens": reported_tokens,
            "history_generated_tokens": history_generated_tokens,
            "solver_note": solver_note,
            "state": run.get("state"),
            "wallclock_s": _safe_float(run.get("final_wallclock_seconds")),
            "actions_per_level": run.get("actions_per_level"),
            "action7_history_count": ids.get("ACTION7", 0),
            "resets_history_count": ids.get("RESET", 0),
            "b0_ref_score": ref[0] if ref else None,
            "b0_ref_levels": ref[1] if ref else None,
            "delta_score_vs_b0": round(score - ref[0], 6) if ref else None,
            "delta_levels_vs_b0": levels - ref[1] if ref else None,
        })
    scores = [r["score"] for r in rows]
    ex = [r["score"] for r in rows if r["prefix"] != "ft09"]
    tokens = [r["reported_generated_tokens"] for r in rows if r["reported_generated_tokens"] is not None]
    states = Counter(str(r["state"]) for r in rows)
    wall = [r["wallclock_s"] for r in rows if r["wallclock_s"] is not None]
    return {
        "aggregate": {
            "games": len(rows),
            "mean_all": round(statistics.fmean(scores), 6) if scores else None,
            "mean_ex_ft09": round(statistics.fmean(ex), 6) if ex else None,
            "median": round(statistics.median(scores), 6) if scores else None,
            "nonzero_games": sum(s > 0 for s in scores),
            "zero_games": sum(s == 0 for s in scores),
            "levels_total": sum(r["levels"] for r in rows),
            "actions_total": sum(r["actions"] for r in rows),
            "reported_generated_tokens_total": sum(tokens) if tokens else None,
            "states": dict(states),
            "all_runs_terminal": all(s in {"won", "gave_up", "cancelled"} for s in states),
            "wallclock_s_mean": round(statistics.fmean(wall), 3) if wall else None,
            "wallclock_s_max": max(wall) if wall else None,
            "target_games_levels": {
                p: next((r["levels"] for r in rows if r["prefix"] == p), None)
                for p in SPARSE_TARGET_GAMES
            },
            "games_above_b0_ref": [
                r["prefix"] for r in rows
                if r["delta_levels_vs_b0"] is not None and r["delta_levels_vs_b0"] > 0
            ],
            "games_below_b0_ref": [
                r["prefix"] for r in rows
                if r["delta_levels_vs_b0"] is not None and r["delta_levels_vs_b0"] < 0
            ],
            "b0_ref_note": "Single Sep-2 B0 orientation only; promotion requires matched-run replication/envelope.",
        },
        "per_game": sorted(rows, key=lambda r: r["game"]),
    }

def _patch_preflight_section():
    return {
        "patch": json.loads((WORKING_DIR / "sparse_p1_patch.json").read_text(encoding="utf-8")),
        "preflight": json.loads((WORKING_DIR / "sparse_p1_preflight.json").read_text(encoding="utf-8")),
    }

def _transcripts_section():
    tdir = WORKING_DIR / "transcripts"
    if not tdir.is_dir():
        return {"available": False, "per_transcript": {}}
    expected = P1_EXPECTED_PROMPT_SHA256
    per = {}
    all_shas = Counter()
    for path in sorted(tdir.glob("*.txt")):
        text = path.read_text(encoding="utf-8", errors="replace")
        blocks = _system_blocks_from_transcript(text)
        shas = [_sha_text(b) for b in blocks]
        all_shas.update(shas)
        per[path.name] = {
            "system_prompt_blocks": len(blocks),
            "distinct_sha256": dict(Counter(shas)),
            "all_blocks_match_p1": bool(blocks) and all(s == expected for s in shas),
            "deleted_probes_present": sorted({
                p for b in blocks for p in P1_DELETED_PROBES if p in b
            }),
            "all_blocks_have_mechanics_marker": bool(blocks) and all(
                P1_MECHANICS_MARKER in b for b in blocks
            ),
            "unknown_action_errors": len(re.findall(r"Unknown action at index", text)),
            "action7_mentions": len(re.findall(r"\bACTION7\b", text)),
            "read_timeout_mentions": len(re.findall(r"ReadTimeout|read timed out|Read timed out", text, re.I)),
            "retryable_mentions": len(re.findall(r"retryable", text, re.I)),
        }
    with_blocks = [v for v in per.values() if v["system_prompt_blocks"] > 0]
    return {
        "available": bool(per),
        "transcripts_total": len(per),
        "transcripts_with_system_prompt": len(with_blocks),
        "system_prompt_blocks_total": sum(v["system_prompt_blocks"] for v in per.values()),
        "distinct_sha256": dict(all_shas),
        "all_blocks_match_p1": bool(with_blocks) and len(with_blocks) == len(per)
            and all(v["all_blocks_match_p1"] for v in with_blocks),
        "no_deleted_probes": bool(with_blocks) and all(not v["deleted_probes_present"] for v in with_blocks),
        "all_blocks_have_mechanics_marker": bool(with_blocks)
            and all(v["all_blocks_have_mechanics_marker"] for v in with_blocks),
        "unknown_action_errors_total": sum(v["unknown_action_errors"] for v in per.values()),
        "read_timeout_mentions_total": sum(v["read_timeout_mentions"] for v in per.values()),
        "retryable_mentions_total": sum(v["retryable_mentions"] for v in per.values()),
        "per_transcript": per,
    }

def _prompt_logs_section():
    pdir = WORKING_DIR / "prompts"
    if not pdir.is_dir():
        return {"available": False, "per_log": {}}
    raw_expected = (WORKING_DIR / "sparse_p1_system_prompt_patched.txt").read_text(encoding="utf-8")
    rendered_expected = _prompt_core(_prompt_log_render(raw_expected))
    expected_rendered_sha = _sha_text(rendered_expected)
    per, shas = {}, Counter()
    for path in sorted(pdir.glob("*.log")):
        text = path.read_text(encoding="utf-8", errors="replace")
        rec = {"system_found": False}
        marker = "[MODEL INPUT]\n[SYSTEM]\n"
        i = text.find(marker)
        if i >= 0:
            start = i + len(marker)
            m = re.search(r"\n\[(USER|ASSISTANT|TOOL RESULT|TOOL|TURN TRANSCRIPT SO FAR)\b", text[start:])
            block = text[start: start + m.start()] if m else text[start:]
            core = _prompt_core(block)
            sha = _sha_text(core)
            shas[sha] += 1
            rec = {
                "system_found": True,
                "sha256": sha,
                "matches_rendered_p1": sha == expected_rendered_sha,
                "deleted_probes_present": [p for p in P1_DELETED_PROBES if p in block],
                "mechanics_marker": P1_MECHANICS_MARKER in block,
            }
        per[path.name] = rec
    found = [v for v in per.values() if v.get("system_found")]
    return {
        "available": bool(per),
        "logs_total": len(per),
        "logs_with_system": len(found),
        "expected_rendered_p1_sha256": expected_rendered_sha,
        "distinct_sha256": dict(shas),
        "all_logs_match_rendered_p1": bool(found) and len(found) == len(per)
            and all(v.get("matches_rendered_p1") is True for v in found),
        "no_deleted_probes": bool(found) and all(not v.get("deleted_probes_present") for v in found),
        "all_have_mechanics_marker": bool(found) and all(v.get("mechanics_marker") is True for v in found),
        "per_log": per,
    }

def _environment_snapshot():
    pkgs = {}
    for name in (
        "torch", "vllm", "transformers", "flashinfer-python", "numpy", "pandas",
        "requests", "arc-agi", "taaf",
    ):
        try:
            pkgs[name] = importlib.metadata.version(name)
        except Exception:
            pkgs[name] = None
    nvidia = {}
    try:
        q = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,uuid,driver_version,memory.total,compute_cap",
                "--format=csv,noheader,nounits",
            ],
            text=True, capture_output=True, timeout=15, check=False,
        )
        nvidia = {"returncode": q.returncode, "stdout": q.stdout.strip(), "stderr": q.stderr.strip()}
    except Exception as exc:
        nvidia = {"error": repr(exc)}
    torch_info = {}
    try:
        import torch
        torch_info = {
            "version": torch.__version__,
            "cuda_runtime": getattr(torch.version, "cuda", None),
            "cuda_available": bool(torch.cuda.is_available()),
            "device_count": int(torch.cuda.device_count()),
            "device_0": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        }
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            torch_info["device_0_total_memory"] = int(props.total_memory)
            torch_info["device_0_capability"] = list(torch.cuda.get_device_capability(0))
    except Exception as exc:
        torch_info = {"error": repr(exc)}
    return {
        "captured_epoch": time.time(),
        "python": sys.version,
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "machine": platform.machine(),
        "sysconfig_platform": sysconfig.get_platform(),
        "packages": pkgs,
        "torch": torch_info,
        "nvidia_smi": nvidia,
        "true_submission": bool(TRUE_SUBMISSION),
        "dataset_sources": list(DATASET_SOURCES),
        "kaggle_input_paths": dict(kaggle_input_paths),
        "selected_env": {
            k: os.environ.get(k) for k in sorted(set(
                list(PUBLIC25_VLLM_PROFILE_ENV)
                + [
                    "LOCAL_ANALYZER_MODEL_ID", "LOCAL_ANALYZER_CONTEXT_WINDOW",
                    "LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS", "LOCAL_ANALYZER_TEMPERATURE",
                    "LOCAL_ANALYZER_TOP_P", "LOCAL_ANALYZER_TOP_K",
                    "LOCAL_ANALYZER_ENABLE_THINKING", "PYTHONPATH",
                ]
            ))
        },
    }

def _fingerprint():
    records = {}
    def add(label, path):
        p = Path(path)
        rec = {"path": str(p), "exists": p.is_file()}
        if p.is_file():
            try:
                rec.update({"bytes": p.stat().st_size, "sha256": _sha_file(p)})
            except Exception as exc:
                rec["error"] = repr(exc)
        records[label] = rec

    for rel in (
        "taaf-kaggle-bundle.json", "setup_commands.json", "teardown_commands.json",
        "serving_setup.py", "benchmark_initial.pkl", "deploy_target.pkl",
    ):
        add(f"bundle/{rel}", BUNDLE_DIR / rel)

    module_names = (
        "inference.framework.solver", "inference.agent.tool_agent", "inference.agent.prompts",
        "inference.agent.action_names", "inference.agent.runtime_state",
        "inference.agent.python_tool_sandbox", "inference.utils.grid_utils",
        "inference.utils.viewer_artifacts",
    )
    for name in module_names:
        try:
            m = importlib.import_module(name)
            add(f"module/{name}", Path(m.__file__).resolve())
        except Exception as exc:
            records[f"module/{name}"] = {"error": repr(exc)}

    # Hash source/config text after gameplay; this cannot steal benchmark time.
    allowed = {".py", ".json", ".jinja", ".toml", ".yaml", ".yml", ".md", ".txt", ".cfg", ".ini"}
    entries, skipped = [], []
    source_root = BUNDLE_DIR / "src"
    if source_root.is_dir():
        for p in sorted(x for x in source_root.rglob("*") if x.is_file()):
            rel = p.relative_to(BUNDLE_DIR).as_posix()
            try:
                size = p.stat().st_size
                if p.suffix.lower() not in allowed:
                    continue
                if size > 16 * 1024 * 1024:
                    skipped.append({"path": rel, "bytes": size, "reason": "source_manifest_file_cap"})
                    continue
                entries.append({"path": rel, "bytes": size, "sha256": _sha_file(p)})
            except Exception as exc:
                skipped.append({"path": rel, "reason": repr(exc)})
    aggregate_payload = json.dumps(entries, sort_keys=True, separators=(",", ":")).encode("utf-8")

    runtime_markers = {}
    for ref, root_s in kaggle_input_paths.items():
        root = Path(root_s)
        found = []
        if root.exists():
            for name in ("PRECONVERTED_MODEL_PROVENANCE.json", "taaf-kaggle-bundle.json"):
                try:
                    for p in list(root.rglob(name))[:8]:
                        found.append({
                            "relative_path": p.relative_to(root).as_posix(),
                            "bytes": p.stat().st_size,
                            "sha256": _sha_file(p),
                            "json": _read_json(p),
                        })
                except Exception as exc:
                    found.append({"search": name, "error": repr(exc)})
        runtime_markers[ref] = {"root": str(root), "exists": root.exists(), "markers": found}

    existing = {}
    for name in (
        "vllm-setup-provenance.json", "vllm-server-identity.json",
        "vllm-server-preflight.json", "vllm-models-final.json",
    ):
        p = WORKING_DIR / name
        if p.is_file():
            existing[name] = {
                "bytes": p.stat().st_size,
                "sha256": _sha_file(p),
                "json": _read_json(p),
            }

    return {
        "captured_epoch": time.time(),
        "critical_files": records,
        "source_text_manifest": {
            "file_count": len(entries),
            "total_bytes": sum(x["bytes"] for x in entries),
            "aggregate_sha256": _sha_bytes(aggregate_payload),
            "files": entries,
            "skipped": skipped,
        },
        "mounted_input_markers": runtime_markers,
        "existing_vllm_provenance": existing,
    }

def _match_game(path_name, game_ids):
    return next((gid for gid in game_ids if gid in path_name), None)

def _load_events_by_game(game_ids):
    out, files = defaultdict(list), []
    art = WORKING_DIR / "artifacts"
    if not art.is_dir():
        return out, files
    for path in sorted(art.rglob("*_events.jsonl")):
        gid = _match_game(path.name, game_ids)
        if gid is None:
            continue
        n = 0
        try:
            for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
                if not line.strip():
                    continue
                try:
                    event = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if isinstance(event, dict):
                    out[gid].append(event)
                    n += 1
            files.append({"path": str(path.relative_to(WORKING_DIR)), "game": gid, "events": n})
        except Exception as exc:
            files.append({"path": str(path.relative_to(WORKING_DIR)), "game": gid, "error": repr(exc)})
    return out, files

def _efficiency_section():
    games = (summary["sections"].get("games") or {}).get("per_game") or []
    game_ids = [r["game"] for r in games]
    events_by_game, event_files = _load_events_by_game(game_ids)
    transcript_dir = WORKING_DIR / "transcripts"
    per = []
    for row in games:
        gid = row["game"]
        evs = events_by_game.get(gid, [])
        analysis_events = [e for e in evs if e.get("type") == "analysis"]
        action_events = [e for e in evs if e.get("type") == "action"]
        model_calls_events = len(analysis_events)

        # Cross-check model call count from transcript system-prompt blocks.
        model_calls_transcript = None
        if transcript_dir.is_dir():
            candidates = [p for p in transcript_dir.glob("*.txt") if gid in p.name]
            if candidates:
                text = candidates[0].read_text(encoding="utf-8", errors="replace")
                model_calls_transcript = len(_system_blocks_from_transcript(text))

        action_steps = {
            int(e["analysis_step"]) for e in action_events
            if isinstance(e.get("analysis_step"), int)
        }
        changed_true = sum(e.get("board_changed") is True for e in action_events)
        changed_false = sum(e.get("board_changed") is False for e in action_events)
        changed_known = changed_true + changed_false
        resets = sum(str(e.get("action_name")) == "RESET" for e in action_events)
        level_complete_actions = sum(e.get("level_completed") is True for e in action_events)
        tokens = row.get("reported_generated_tokens")
        levels = row.get("levels") or 0
        actions = row.get("actions") or 0
        mc = model_calls_transcript if model_calls_transcript is not None else model_calls_events
        per.append({
            "game": gid,
            "actions": actions,
            "levels": levels,
            "reported_generated_tokens": tokens,
            "model_calls_from_transcript": model_calls_transcript,
            "analysis_events": model_calls_events,
            "action_bearing_analysis_steps": len(action_steps),
            "action_events": len(action_events),
            "board_changed_true": changed_true,
            "board_changed_false": changed_false,
            "board_changed_known": changed_known,
            "no_change_share": round(changed_false / changed_known, 6) if changed_known else None,
            "resets_in_action_events": resets,
            "level_complete_action_events": level_complete_actions,
            "actions_per_model_call": round(actions / mc, 6) if mc else None,
            "actions_per_action_bearing_call": round(actions / len(action_steps), 6) if action_steps else None,
            "tokens_per_action": round(tokens / actions, 6) if tokens is not None and actions else None,
            "tokens_per_level": round(tokens / levels, 6) if tokens is not None and levels else None,
        })
    total_actions = sum(r["actions"] for r in per)
    total_levels = sum(r["levels"] for r in per)
    token_values = [r["reported_generated_tokens"] for r in per if r["reported_generated_tokens"] is not None]
    total_tokens = sum(token_values) if token_values else None
    calls = [r["model_calls_from_transcript"] for r in per if r["model_calls_from_transcript"] is not None]
    total_calls = sum(calls) if calls else None
    bearing = sum(r["action_bearing_analysis_steps"] for r in per)
    change_true = sum(r["board_changed_true"] for r in per)
    change_false = sum(r["board_changed_false"] for r in per)
    change_known = change_true + change_false
    return {
        "data_quality": {
            "event_files": event_files,
            "games_with_event_data": sum(bool(events_by_game.get(gid)) for gid in game_ids),
            "games_with_transcript_call_count": len(calls),
            "note": "Model-call count = number of transcript SYSTEM PROMPT blocks. Action-bearing calls use viewer action-event analysis_step.",
        },
        "aggregate": {
            "actions": total_actions,
            "levels": total_levels,
            "reported_generated_tokens": total_tokens,
            "model_calls": total_calls,
            "action_bearing_model_calls": bearing if event_files else None,
            "actions_per_model_call": round(total_actions / total_calls, 6) if total_calls else None,
            "actions_per_action_bearing_call": round(total_actions / bearing, 6) if bearing else None,
            "tokens_per_action": round(total_tokens / total_actions, 6) if total_tokens is not None and total_actions else None,
            "tokens_per_model_call": round(total_tokens / total_calls, 6) if total_tokens is not None and total_calls else None,
            "levels_per_action": round(total_levels / total_actions, 8) if total_actions else None,
            "tokens_per_level": round(total_tokens / total_levels, 6) if total_tokens is not None and total_levels else None,
            "board_changed_true": change_true,
            "board_changed_false": change_false,
            "no_change_share": round(change_false / change_known, 6) if change_known else None,
            "resets_in_action_events": sum(r["resets_in_action_events"] for r in per),
            "unknown_action_errors": (summary["sections"].get("transcripts") or {}).get("unknown_action_errors_total"),
        },
        "per_game": per,
    }

_HEALTH_PATTERNS = {
    "read_timeout": re.compile(r"ReadTimeout|read timed out|Read timed out", re.I),
    "connect_timeout": re.compile(r"ConnectTimeout|connect timed out", re.I),
    "generic_timeout": re.compile(r"\btimeout\b|\btimed out\b", re.I),
    "retryable": re.compile(r"\bretryable\b", re.I),
    "http_5xx": re.compile(r"(?:HTTP|status(?:_code)?)\D{0,12}5\d\d", re.I),
    "traceback": re.compile(r"Traceback \(most recent call last\)", re.I),
    "cuda_oom": re.compile(r"CUDA.*out of memory|OutOfMemoryError|CUDA OOM", re.I),
    "watchdog_restart": re.compile(r"watchdog.{0,80}restart|restart.{0,80}watchdog", re.I),
    "health_failure": re.compile(r"health.{0,40}(?:fail|unhealthy)|unhealthy", re.I),
}

def _json_key_values(obj, terms, prefix=""):
    found = {}
    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            if any(t in str(k).lower() for t in terms) and not isinstance(v, (dict, list)):
                found[key] = v
            found.update(_json_key_values(v, terms, key))
    elif isinstance(obj, list):
        for i, v in enumerate(obj[:200]):
            found.update(_json_key_values(v, terms, f"{prefix}[{i}]"))
    return found

def _runtime_health_section():
    scan_files = []
    for p in sorted(WORKING_DIR.glob("*.log")):
        scan_files.append(p)
    tdir = WORKING_DIR / "transcripts"
    if tdir.is_dir():
        scan_files.extend(sorted(tdir.glob("*.txt")))
    counts = Counter()
    per_file = {}
    samples = defaultdict(list)
    for p in scan_files:
        try:
            if p.stat().st_size > 64 * 1024 * 1024:
                continue
            text = p.read_text(encoding="utf-8", errors="replace")
        except Exception:
            continue
        local = {}
        for name, rx in _HEALTH_PATTERNS.items():
            n = len(rx.findall(text))
            if n:
                local[name] = n
                counts[name] += n
                if len(samples[name]) < 12:
                    for line in text.splitlines():
                        if rx.search(line):
                            samples[name].append({
                                "file": str(p.relative_to(WORKING_DIR)),
                                "line": line[:500],
                            })
                            if len(samples[name]) >= 12:
                                break
        if local:
            per_file[str(p.relative_to(WORKING_DIR))] = local

    json_artifacts = {}
    for name in (
        "vllm-watchdog-status.json", "vllm-server-teardown.json",
        "vllm-server-identity.json", "vllm-server-preflight.json",
        "vllm-setup-provenance.json", "vllm-models-final.json",
    ):
        p = WORKING_DIR / name
        if p.is_file():
            data = _read_json(p)
            json_artifacts[name] = {
                "bytes": p.stat().st_size,
                "sha256": _sha_file(p),
                "selected_health_fields": _json_key_values(
                    data, ("restart", "failure", "fail", "timeout", "health", "error", "attempt")
                ) if data is not None else {},
                "json": data,
            }

    prom = WORKING_DIR / "vllm-metrics-final.prom"
    prom_summary = {"exists": prom.is_file()}
    if prom.is_file():
        text = prom.read_text(encoding="utf-8", errors="replace")
        interesting = [
            line for line in text.splitlines()
            if re.search(r"specul|mtp|accept|token|throughput|cache|request|preempt|latency", line, re.I)
        ]
        prom_summary.update({
            "bytes": prom.stat().st_size,
            "sha256": _sha_file(prom),
            "interesting_lines": interesting[:1000],
            "interesting_lines_truncated": len(interesting) > 1000,
        })

    g = (summary["sections"].get("games") or {}).get("aggregate") or {}
    states = g.get("states") or {}
    return {
        "pattern_counts": dict(counts),
        "pattern_samples": dict(samples),
        "per_file_counts": per_file,
        "json_artifacts": json_artifacts,
        "vllm_metrics_final": prom_summary,
        "game_states": states,
        "crashed_games": int(states.get("crashed", 0) or 0),
        "cancelled_games": int(states.get("cancelled", 0) or 0),
        "warning": "Pattern counts are forensic indicators, not all semantically distinct failures.",
    }

def _timing_section():
    pre = _read_json(WORKING_DIR / "sparse_p1_preflight.json") or {}
    now = time.time()
    game_rows = (summary["sections"].get("games") or {}).get("per_game") or []
    wall = [r["wallclock_s"] for r in game_rows if r.get("wallclock_s") is not None]
    pre_epoch = pre.get("preflight_completed_epoch")
    start_epoch = pre.get("notebook_start_epoch", NOTEBOOK_START_EPOCH)
    return {
        "notebook_start_epoch": start_epoch,
        "preflight_completed_epoch": pre_epoch,
        "post_teardown_forensic_capture_epoch": now,
        "startup_to_preflight_s": round(pre_epoch - start_epoch, 6)
            if isinstance(pre_epoch, (int, float)) else None,
        "run_cell_plus_teardown_s": round(now - pre_epoch, 6)
            if isinstance(pre_epoch, (int, float)) else None,
        "notebook_start_to_forensic_capture_s": round(now - start_epoch, 6)
            if isinstance(start_epoch, (int, float)) else None,
        "per_game_wallclock_available": len(wall),
        "per_game_wallclock_mean_s": round(statistics.fmean(wall), 3) if wall else None,
        "per_game_wallclock_max_s": max(wall) if wall else None,
        "note": "run_cell_plus_teardown_s starts immediately after preflight and includes game-list setup, benchmark, watchdog stop and teardown.",
    }

def _verdict():
    g = (summary["sections"].get("games") or {}).get("aggregate") or {}
    pp = summary["sections"].get("patch_preflight") or {}
    patch, pre = pp.get("patch") or {}, pp.get("preflight") or {}
    tr = summary["sections"].get("transcripts") or {}
    pl = summary["sections"].get("prompt_logs") or {}
    hard = {
        "benchmark_has_25_games": g.get("games") == 25,
        "all_runs_terminal": g.get("all_runs_terminal") is True,
        "actions_nonzero": (g.get("actions_total") or 0) > 0,
        "patch_is_frozen_b0": patch.get("base_is_frozen_b0") is True,
        "patch_is_arm_b_plus_c": patch.get("treatment") == "donor Arm B + Arm C only",
        "action7_not_changed": patch.get("action7_change_included") is False
            and patch.get("action7_to_engine_after") is None,
        "preflight_all_true": bool(pre.get("checks"))
            and all(v is True for v in (pre.get("checks") or {}).values()),
        "transcripts_available": tr.get("available") is True,
        "transcripts_cover_25_games": tr.get("transcripts_with_system_prompt") == 25,
        "all_transcript_prompt_blocks_match_p1": tr.get("all_blocks_match_p1") is True,
        "transcripts_have_no_deleted_probes": tr.get("no_deleted_probes") is True,
        "transcripts_have_mechanics_marker": tr.get("all_blocks_have_mechanics_marker") is True,
    }
    optional = {
        "prompt_logs_available": pl.get("available") is True,
        "prompt_logs_match_rendered_p1": pl.get("all_logs_match_rendered_p1")
            if pl.get("available") else None,
        "prompt_logs_have_no_deleted_probes": pl.get("no_deleted_probes")
            if pl.get("available") else None,
        "prompt_logs_have_mechanics_marker": pl.get("all_have_mechanics_marker")
            if pl.get("available") else None,
    }
    return {
        "hard_gates": hard,
        "optional_crosscheck": optional,
        "attributable_to_p1": all(v is True for v in hard.values()),
    }

# Order matters for derived sections.
_section("games", _games_section)
_section("patch_preflight", _patch_preflight_section)
_section("transcripts", _transcripts_section)
_section("prompt_logs", _prompt_logs_section)
_section("efficiency", _efficiency_section)
_section("environment", _environment_snapshot)
_section("runtime_fingerprint", _fingerprint)
_section("runtime_health", _runtime_health_section)
_section("timing", _timing_section)
_section("verdict", _verdict)

# Separate small files make later comparisons easy without loading the large master summary.
for sec, filename in (
    ("environment", "p1_environment.json"),
    ("runtime_fingerprint", "p1_runtime_fingerprint.json"),
    ("runtime_health", "p1_runtime_health.json"),
    ("efficiency", "p1_efficiency.json"),
    ("timing", "p1_run_timing.json"),
):
    if sec in summary["sections"]:
        _write_json(filename, summary["sections"][sec])

limits = """# P1 evidence limits

- Raw Jupyter/Kaggle cell stdout is NOT tee'd into the ZIP. Tee-ing stdout would alter the execution path.
  Experiment-critical console facts are mirrored in patch/preflight/runtime JSON. If a cell itself errors or
  shows an unexpected warning, also keep the executed Kaggle notebook as supplemental evidence.
- `requests.jsonl` is intentionally absent because frozen B0 uses `save_request_logs=False`.
  `transcripts/` proves every system-prompt block and `prompts/` preserves Duck's latest rendered model-input
  snapshot per game.
- Promotion still requires >=2 matched runs; a single historical B0 reference is orientation, not an envelope.
"""
(WORKING_DIR / "P1_SPARSE_EVIDENCE_LIMITS.md").write_text(limits, encoding="utf-8")

out = WORKING_DIR / "sparse_p1_postrun_summary.json"
out.write_text(json.dumps(summary, indent=2, sort_keys=True, default=str) + "\n", encoding="utf-8")

g = (summary["sections"].get("games") or {}).get("aggregate") or {}
e = (summary["sections"].get("efficiency") or {}).get("aggregate") or {}
v = summary["sections"].get("verdict") or {}
print(
    "P1_POSTRUN_V2 "
    f"games={g.get('games')} mean={g.get('mean_all')} ex_ft09={g.get('mean_ex_ft09')} "
    f"levels={g.get('levels_total')} nonzero={g.get('nonzero_games')} "
    f"actions={g.get('actions_total')} tokens={g.get('reported_generated_tokens_total')} "
    f"model_calls={e.get('model_calls')} no_change_share={e.get('no_change_share')} "
    f"attributable={v.get('attributable_to_p1')}",
    flush=True,
)
if summary["warnings"]:
    print("P1_POSTRUN_WARNINGS", summary["warnings"], flush=True)


P1_POSTRUN_V2 games=25 mean=6.911072 ex_ft09=5.214907 levels=35 nonzero=20 actions=3882 tokens=1979584 model_calls=1340 no_change_share=0.05899 attributable=True


## 10. Package and self-check the P1 evidence bundle

Writes `/kaggle/working/P1_SPARSE_RUN_EVIDENCE.zip`. Critical files are packed first; vLLM's final
Prometheus metrics are included; source data are capped at 240 MiB. The ZIP then checks itself:
for a public-25 run, all 25 raw transcripts must physically be inside the ZIP and no tier-0/1/2
critical file may have been dropped. Packaging remains non-fatal to gameplay, but the offline verifier
will reject an incomplete evidence ZIP.


In [12]:
# P1 SPARSE EVIDENCE V2 package + physical completeness check. Non-fatal to gameplay.
import hashlib
import json
import traceback
import zipfile
from pathlib import Path

P1_RUN_ZIP = WORKING_DIR / "P1_SPARSE_RUN_EVIDENCE.zip"
P1_ZIP_MANIFEST = WORKING_DIR / "P1_SPARSE_RUN_EVIDENCE_MANIFEST.json"
P1_ZIP_SELFCHECK = WORKING_DIR / "P1_SPARSE_RUN_EVIDENCE_SELFCHECK.json"
P1_ZIP_ERROR = WORKING_DIR / "P1_SPARSE_RUN_EVIDENCE_ERROR.json"
P1_MAX_SINGLE_FILE_BYTES = 48 * 1024 * 1024
P1_MAX_SOURCE_BYTES = 240 * 1024 * 1024

TIERS = [
    (0, [
        "sparse_p1_patch.json", "sparse_p1_preflight.json", "sparse_p1_postrun_summary.json",
        "sparse_p1_MANIFEST.md", "sparse_p1_system_prompt_base.txt",
        "sparse_p1_system_prompt_patched.txt", "sparse_p1_prompt_diff.txt",
        "p1_environment.json", "p1_runtime_fingerprint.json", "p1_runtime_health.json",
        "p1_efficiency.json", "p1_run_timing.json", "P1_SPARSE_EVIDENCE_LIMITS.md",
    ]),
    (1, ["benchmark.json", "score.json", "submission.parquet", "taaf_setup_env.json"]),
    (2, ["transcripts", "prompts"]),
    (3, ["artifacts"]),
    (4, [
        "vllm-openai-server.log", "vllm_server.log", "server.log",
        "vllm-server-identity.json", "vllm-server-preflight.json", "vllm-server-teardown.json",
        "vllm-setup-provenance.json", "vllm-watchdog-status.json", "vllm-models-final.json",
        "vllm-metrics-final.prom", "git_status.txt",
    ]),
    (5, ["solver_analysis"]),
    (6, ["server_recording"]),
]
TRUE_SUBMISSION_TIERS = {0, 1}

def _candidates():
    out = []
    for tier, names in TIERS:
        if TRUE_SUBMISSION and tier not in TRUE_SUBMISSION_TIERS:
            continue
        for name in names:
            p = WORKING_DIR / name
            if p.is_file():
                out.append((tier, p.relative_to(WORKING_DIR).as_posix(), p))
            elif p.is_dir():
                for f in sorted(x for x in p.rglob("*") if x.is_file()):
                    out.append((tier, f.relative_to(WORKING_DIR).as_posix(), f))
    if not TRUE_SUBMISSION:
        already = {a for _, a, _ in out}
        for pattern in (
            "*.json", "*.jsonl", "*.ndjson", "*.txt", "*.log", "*.md", "*.csv",
            "*.parquet", "*.prom",
        ):
            for p in sorted(WORKING_DIR.glob(pattern)):
                if p in {P1_RUN_ZIP, P1_ZIP_MANIFEST, P1_ZIP_SELFCHECK, P1_ZIP_ERROR}:
                    continue
                rel = p.relative_to(WORKING_DIR).as_posix()
                if rel not in already:
                    out.append((9, rel, p))
    return sorted(out, key=lambda x: (x[0], x[1]))

def _select():
    selected, skipped, seen = [], [], set()
    total = 0
    for tier, arcname, path in _candidates():
        try:
            rp = path.resolve()
            if rp in seen:
                continue
            size = path.stat().st_size
        except OSError as exc:
            skipped.append({"path": arcname, "reason": f"stat:{type(exc).__name__}", "tier": tier})
            continue
        if size > P1_MAX_SINGLE_FILE_BYTES:
            skipped.append({"path": arcname, "bytes": size, "reason": "single_file_cap", "tier": tier})
            continue
        if total + size > P1_MAX_SOURCE_BYTES:
            skipped.append({"path": arcname, "bytes": size, "reason": "aggregate_cap", "tier": tier})
            continue
        selected.append((tier, arcname, path, size))
        seen.add(rp)
        total += size
    return selected, skipped, total

def _physical_selfcheck(zip_path, selected, skipped):
    with zipfile.ZipFile(zip_path) as zf:
        names = set(zf.namelist())
        corrupt = zf.testzip()
    required = {
        "sparse_p1_patch.json", "sparse_p1_preflight.json", "sparse_p1_postrun_summary.json",
        "p1_environment.json", "p1_runtime_fingerprint.json", "p1_runtime_health.json",
        "p1_efficiency.json", "p1_run_timing.json", "benchmark.json",
    }
    if not TRUE_SUBMISSION:
        required.add("score.json")
    missing = sorted(required - names)
    transcript_names = sorted(
        n for n in names if n.startswith("transcripts/") and n.endswith(".txt")
    )
    prompt_names = sorted(
        n for n in names if n.startswith("prompts/") and n.endswith(".log")
    )
    critical_skips = [
        s for s in skipped if isinstance(s.get("tier"), int) and s["tier"] <= 2
    ]
    checks = {
        "zip_integrity": corrupt is None,
        "required_files_present": not missing,
        "no_critical_tier_0_1_2_skips": not critical_skips,
        "public25_has_25_raw_transcripts": True if TRUE_SUBMISSION else len(transcript_names) == 25,
    }
    # Prompt snapshots are a secondary cross-check; transcripts are the attribution hard gate.
    optional = {
        "prompt_log_count": len(prompt_names),
        "public25_has_25_prompt_logs": None if TRUE_SUBMISSION else len(prompt_names) == 25,
        "vllm_metrics_final_prom_in_zip": "vllm-metrics-final.prom" in names,
    }
    return {
        "evidence_schema_version": P1_EVIDENCE_SCHEMA_VERSION,
        "checks": checks,
        "optional": optional,
        "hard_complete": all(v is True for v in checks.values()),
        "missing_required": missing,
        "critical_skips": critical_skips,
        "raw_transcript_count": len(transcript_names),
        "prompt_log_count": len(prompt_names),
        "zip_corrupt_member": corrupt,
    }

def _package():
    selected, skipped, total = _select()
    if P1_RUN_ZIP.exists():
        P1_RUN_ZIP.unlink()

    manifest_included = []
    with zipfile.ZipFile(P1_RUN_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for tier, arcname, path, _ in selected:
            try:
                data = path.read_bytes()
            except OSError as exc:
                skipped.append({"path": arcname, "reason": f"read:{type(exc).__name__}", "tier": tier})
                continue
            zf.writestr(arcname, data)
            manifest_included.append({
                "path": arcname, "tier": tier, "bytes": len(data),
                "sha256": hashlib.sha256(data).hexdigest(),
            })

    selfcheck = _physical_selfcheck(P1_RUN_ZIP, selected, skipped)
    P1_ZIP_SELFCHECK.write_text(
        json.dumps(selfcheck, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    selfcheck_data = P1_ZIP_SELFCHECK.read_bytes()
    with zipfile.ZipFile(P1_RUN_ZIP, "a", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        zf.writestr(P1_ZIP_SELFCHECK.name, selfcheck_data)
    manifest_included.append({
        "path": P1_ZIP_SELFCHECK.name, "tier": 0, "bytes": len(selfcheck_data),
        "sha256": hashlib.sha256(selfcheck_data).hexdigest(),
    })

    manifest = {
        "experiment": P1_NAME,
        "patch_version": P1_PATCH_VERSION,
        "evidence_schema_version": P1_EVIDENCE_SCHEMA_VERSION,
        "true_submission": bool(TRUE_SUBMISSION),
        "source_cap_bytes": P1_MAX_SOURCE_BYTES,
        "source_bytes_selected_before_manifest": total,
        "manifest_self_hash_excluded_by_design": True,
        "included": manifest_included,
        "skipped": skipped,
        "selfcheck_hard_complete": selfcheck["hard_complete"],
    }
    P1_ZIP_MANIFEST.write_text(
        json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    manifest_data = P1_ZIP_MANIFEST.read_bytes()
    with zipfile.ZipFile(P1_RUN_ZIP, "a", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        zf.writestr(P1_ZIP_MANIFEST.name, manifest_data)

    # Final corruption test after appending selfcheck + manifest.
    with zipfile.ZipFile(P1_RUN_ZIP) as zf:
        final_corrupt = zf.testzip()
    return manifest, selfcheck, final_corrupt

try:
    m, sc, corrupt = _package()
    print(
        f"P1_EVIDENCE_ZIP_V2 path={P1_RUN_ZIP} zip_bytes={P1_RUN_ZIP.stat().st_size} "
        f"files={len(m['included'])+1} source_bytes={m['source_bytes_selected_before_manifest']} "
        f"skipped={len(m['skipped'])} hard_complete={sc['hard_complete']} corrupt={corrupt}",
        flush=True,
    )
    if not sc["hard_complete"]:
        print(
            "P1_EVIDENCE_ZIP_INCOMPLETE: gameplay result may still be valid, but do not use this ZIP "
            "as self-contained P1 evidence until the missing/critical-skipped items are recovered.",
            flush=True,
        )
except Exception as exc:
    err = {"error": repr(exc), "traceback": traceback.format_exc()}
    try:
        P1_ZIP_ERROR.write_text(json.dumps(err, indent=2) + "\n", encoding="utf-8")
    except Exception:
        pass
    print(f"P1_EVIDENCE_ZIP_NONFATAL_ERROR {exc!r}", flush=True)


P1_EVIDENCE_ZIP_V2 path=/kaggle/working/P1_SPARSE_RUN_EVIDENCE.zip zip_bytes=19876857 files=154 source_bytes=171975520 skipped=0 hard_complete=True corrupt=None
